#### Loading packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


#### Setting root directory path

In [ ]:
ROOT = r'C:\Users\PC_DS_ECON_5\Desktop\data-analytics-python'


#### Data source used: NLSY97

The NLSY97 is a `longitudinal` (i.e. `panel`) survey of individuals born between 1980 and 1984. The first round of data collection took place in 1997 and was followed by additional survey rounds every one or two years.


#### Loading data

Reminder: We need the `pyarrow` package to import a `.parquet` datafile.

In [ ]:
nlsy97_income_hours_all_df = pd.read_parquet(ROOT + '/data/nlsy97_income_hours_all_df.parquet')
nlsy97_additional_vars_df = pd.read_parquet(ROOT + '/data/nlsy97_additional_vars_df.parquet')


#### Dictionary of variable labels

In [ ]:
long_varlabels_dict = {
    'gender': 'Gender',
    'ethnicity': 'Ethnicity',
    'educ': 'Highest Grade Completed',
    'degree': 'Highest Degree Earned',
    'meduc_bio': 'Highest Grade Completed (Biological Mother)',
    'meduc_res': 'Highest Grade Completed (Residential Mother)',
    'feduc_bio': 'Highest Grade Completed (Biological Father)',
    'feduc_res': 'Highest Grade Completed (Residential Father)',
    'educ_num': 'Inferred Years of Education',
    'educ_cat': 'Highest Degree Earned',
    'meduc_bio_num': 'Inferred Years of Education (Biological Mother)',
    'meduc_res_num': 'Inferred Years of Education (Residential Mother)',
    'feduc_bio_num': 'Inferred Years of Education (Biological Father)',
    'feduc_res_num': 'Inferred Years of Education (Residential Father)',
    'meduc_bio_cat': 'Inferred Degree (Biological Mother)',
    'meduc_res_cat': 'Inferred Degree (Residential Mother)',
    'feduc_bio_cat': 'Inferred Degree (Biological Father)',
    'feduc_res_cat': 'Inferred Degree (Residential Father)',
    'hh_yinc_1997': 'Household Income in 1997',
    'hh_yinc_source_1997': 'Source of Household Income in 1997', 
    'hh_net_worth_1997': 'Household Net Worth in 1997',
    'ASVAB_score': 'ASVAB score percentile (x1000)',
    'ASVAB_pct': 'ASVAB score percentile rank'
}


short_varlabels_dict = {
    'gender': 'Gender',
    'ethnicity': 'Ethnicity',
    'educ': 'Highest Grade Completed',
    'degree': 'Highest Degree Earned',
    'meduc_bio': 'Highest Grade Compl. (Biol. Mother)',
    'meduc_res': 'Highest Grade Compl. (Resid. Mother)',
    'feduc_bio': 'Highest Grade Compl. (Biol. Father)',
    'feduc_res': 'Highest Grade Compl. (Resid. Father)',
    'educ_num': 'Inferred Years of Educ.',
    'educ_cat': 'Highest Degree Earned',
    'meduc_bio_num': 'Inferred Years of Educ. (Biol. Mother)',
    'meduc_res_num': 'Inferred Years of Educ. (Resid. Mother)',
    'feduc_bio_num': 'Inferred Years of Educ. (Biol. Father)',
    'feduc_res_num': 'Inferred Years of Educ. (Resid. Father)',
    'meduc_bio_cat': 'Inferred Highest Degree (Biol. Mother)',
    'meduc_res_cat': 'Inferred Highest Degree (Resid. Mother)',
    'feduc_bio_cat': 'Inferred Highest Degree (Biol. Father)',
    'feduc_res_cat': 'Inferred Highest Degree (Resid. Father)',
    'hh_yinc_1997': 'HH Income in 1997',
    'hh_yinc_source_1997': 'Source of HH Income in 1997', 
    'hh_net_worth_1997': 'HH Net Worth in 1997',
    'ASVAB_score': 'ASVAB score ptile (x1000)',
    'ASVAB_pct': 'ASVAB score ptile rank'
}

##### Preparing the variables

As in the previous section on correlation, we will use the following variables:

- log hourly pay (in 2023)
- educational attainment categories (highest degree / grade completed)
- ASVAB test score percentile rank (in 1997)

First, let us construct hourly pay `hpay` from annual income and annual hours worked in 2023. We then calculate the natural logarithm of hourly pay, `log_hpay`.

In [ ]:
df = nlsy97_income_hours_all_df.copy()

df = df.loc[df['year'].eq(2023),['person_id', 'yinc', 'yhours']].copy()

valid = (
    df['yinc'].notna() &
    df['yhours'].notna() &
    df['yinc'].ne(0) &
    df['yhours'].ne(0)
)

df['hpay'] = np.where(valid,
                      df['yinc'] / df['yhours'],
                      pd.NA)

df = df.convert_dtypes()
df['log_hpay'] = np.where(df['hpay'].gt(0) & df['hpay'].notna(),
                          np.log(df['hpay']),
                          pd.NA)
df = df.convert_dtypes()
inc_df = df.copy()
inc_df

Second, let us use the same educational attainment categories as in previous examples.

For study participants, we construct the educational attainment categories in the following way:

$$
educ\_cat = 
\begin{cases}
\text{Less than High School} & \text{if } degree = \text{No degree} \text{ or } degree = \text{GED}\\
\text{High School} & \text{if } degree = \text{High school} \text{ and } educ\_num \leq 12\\
\text{Some College} & \text{if } (degree = \text{High school} \text{ and } educ\_num > 12) \text{ or } degree = \text{Junior college}\\
\text{Bachelor Plus} & \text{if } degree \in \{\text{Bachelor}, \text{Master}, \text{Professional}, \text{PhD}\}
\end{cases}
$$

For parents, we construct the educational attainment categories in the following way:

$$
[m/f]educ\_cat = 
\begin{cases}
\text{Less than High School} & \text{if } [m/f]educ\_num \leq 11\\
\text{High School} & \text{if } [m/f]educ\_num = 12\\
\text{Some College} & \text{if } [m/f]educ\_num \in [13,15] \\
\text{Bachelor Plus} & \text{if } [m/f]educ\_num \geq 16
\end{cases}
$$

In [ ]:
df = nlsy97_additional_vars_df.copy()
varnames = ['meduc_bio', 'meduc_res', 'feduc_bio', 'feduc_res']
categories = ['Less than High School', 'High School', 'Some College', 'Bachelor Plus']
for varname in varnames:
    df[varname + '_cat'] = pd.NA 
    df.loc[df[varname + '_num'].lt(12), varname + '_cat'] = categories[0]
    df.loc[df[varname + '_num'].eq(12), varname + '_cat'] = categories[1]
    df.loc[df[varname + '_num'].gt(12) & df[varname + '_num'].lt(16), varname + '_cat'] = categories[2]
    df.loc[df[varname + '_num'].ge(16), varname + '_cat'] = categories[3]
    df[varname + '_cat'] = pd.Categorical(df[varname + '_cat'], categories=categories, ordered=True)


df['educ_cat'] = pd.NA
df.loc[df['degree'].eq('No degree'), 'educ_cat'] = categories[0]
df.loc[df['degree'].eq('GED'), 'educ_cat'] = categories[0]
df.loc[df['degree'].eq('High school') & df['educ_num'].le(12), 'educ_cat'] = categories[1]
df.loc[df['degree'].eq('High school') & df['educ_num'].gt(12), 'educ_cat'] = categories[2]
df.loc[df['degree'].eq('Junior college'), 'educ_cat'] = categories[2]
df.loc[df['degree'].eq('Bachelor'), 'educ_cat'] = categories[3]
df.loc[df['degree'].eq('Master'), 'educ_cat'] = categories[3]
df.loc[df['degree'].eq('Professional'), 'educ_cat'] = categories[3]
df.loc[df['degree'].eq('PhD'), 'educ_cat'] = categories[3]
df['educ_cat'] = pd.Categorical(df['educ_cat'], categories=categories, ordered=True)

nlsy97_additional_vars_df = df.copy()

Third, let us prepare the variable `ASVAB_pct`, which measures each participant's ASVAB test score percentile rank in 1997.

In [ ]:
df = nlsy97_additional_vars_df.copy()
df['ASVAB_pct'] = df['ASVAB_score'] / 1000
df = df.convert_dtypes()
nlsy97_additional_vars_df = df.copy()

Finally, we merge the data frame `nlsy97_additional_vars_df` containing the demographic, test percentile, and educational variables with the data frame `inc_df` containing hourly pay and log hourly pay.

In [ ]:
merged_df = pd.merge(left=nlsy97_additional_vars_df,
                     right=inc_df,
                     on='person_id',
                     how='left')

#### Simple linear regression

##### Scatterplot of log hourly pay and ASVAB test score percentile rank

Let us begin with a scatter plot of log hourly pay against ASVAB test score percentile rank.

In [ ]:
df = merged_df.copy()

fig, ax = plt.subplots()
ax.scatter(x=df['ASVAB_pct'], y=df['log_hpay'], s=2, alpha=0.5)
# yticksvalues = [1, 10, 100, 1000]
# yticksvalues_log = np.log(yticksvalues)
# yticksvalues_labels = np.array(yticksvalues).astype(str)
# ax.set_yticks(ticks=yticksvalues_log, labels=yticksvalues_labels)
ax.set_ylabel('Hourly Pay in 2023 (dollars, logarithmic scale)', fontsize=10)
ax.set_xlabel('ASVAB test score percentile rank in 1997')



Visually, there appears to be a positive relationship: individuals with higher ASVAB percentile ranks tend to have higher hourly pay.

The relationship is far from deterministic. There is substantial variation in hourly pay even among individuals with similar ASVAB scores.

To visualise the average relationship more clearly, let us consider the conditional mean of log hourly pay by ASVAB percentile rank bins, as we have done before.

In [ ]:
df = merged_df.copy()

df['ASVAB_pct_bin'] = np.floor(df['ASVAB_pct'])


plot_df = df.groupby(by='ASVAB_pct_bin').agg(mean_log_hpay = ('log_hpay', 'mean')).reset_index().copy()

plot_df = plot_df.convert_dtypes()
plot_df
fig, ax = plt.subplots()

ax.scatter(x=plot_df['ASVAB_pct_bin'], y=plot_df['mean_log_hpay'], s=10)

y_tickvalues = np.array([10,30,100])
y_tickvalues_log = np.log(y_tickvalues) 
y_tickvalues_str = y_tickvalues.astype(str)
ax.yaxis.set_ticks(ticks=y_tickvalues_log, labels=y_tickvalues_str)

ax.set_ylabel('Conditional geometric mean of hourly pay in 2023 (dollars)', fontsize=8)
ax.set_xlabel('ASVAB test score percentile rank in 1997')


The conditional mean plot confirms the positive association, but points to a non-linear relationship.

The population counterpart of the conditional mean plot is the conditional mean function:

$$
\mathbb{E}[log\_hpay \mid ASVAB\_pct]
$$

**Linear regression as linear approximation of the conditional mean function**

Linear regression approximates the conditional mean function using a straight line. Although the true conditional mean function may be non-linear, a **linear approximation** often provides a useful summary of the **average relationship** between two variables.

**Simple** linear regression means that there is **only one** conditioning variable, called **regressor**.

In the case of simple linear regression, the approximation corresponds to:

$$
\mathbb{E}[Y \mid X]\ \approx\ \overbrace{\beta_0}^{\mathclap{\substack{\text{intercept}\\ \text{parameter}}}}  
\ \ +\ \ \overbrace{\beta_1}^{\mathclap{\substack{\text{slope}\\ \text{coefficient}\\ \text{parameter}}}} 
\cdot \underbrace{X}_{\mathclap{\substack{\text{regressor}}}} 
$$

**Ordinary Least Squares in the case of Simple Linear Regression**

What is the best-fitting linear function? 
 - We need a criteria: **minimising a loss function**.
 - The **most common** loss criterion is the **mean of squared errors**:

$$
(\beta_0, \beta_1) = \arg \underset{(b_0, b_1)}{\min} \  \mathbb{E}\left[\left(Y - (b_0 + b_1 \cdot X)\right)^2\right] 
$$

This minimisation is the population version of the **Ordinary Least Squares (OLS)** problem. The use of squared errors is not arbitrary. In fact, the population mean can be characterized as the value that minimizes the expected squared deviation:

$$
\mathbb{E}[Y] = \arg \underset{m}{\min} \ \mathbb{E}\left[\left(Y - m\right)^2\right]
$$

Linear regression extends this idea from finding the best constant predictor to **finding the best linear predictor**.

In the population, the deviation between the value of the outcome variable and the linear prediction for the conditional mean is called the **error term**:

$$
\underbrace{\varepsilon}_{\mathclap{\substack{\text{error}\\ \text{term}}}}\ =\ \underbrace{Y}_{\mathclap{\substack{\text{outcome}}}} - \underbrace{(\beta_0 + \beta_1 \cdot X)}_{\mathclap{\substack{\text{linear prediction}}}} 
\ \ \ \ \ \Longleftrightarrow \ \ \ \ \  
\underbrace{Y}_{\mathclap{\substack{\text{outcome}}}}\ =\ \underbrace{\beta_0 + \beta_1 \cdot X}_{\mathclap{\substack{\text{linear prediction}}}} + \underbrace{\varepsilon}_{\mathclap{\substack{\text{error}\\ \text{term}}}}
$$

In the case of simple linear regression, after solving the OLS minimisation, we obtain the following results for the parameter values of the best linear approximation:

$$
\begin{array}{rcll}
\beta_0 &=& \mathbb{E}[Y] - \beta_1 \cdot \mathbb{E}[X] & \ \ (\text{intercept}) \\ \\
\beta_1 &=& \displaystyle \frac{Cov(X,Y)}{Var(X)}                     &  \ \ (\text{slope coefficient})
\end{array}
$$

The slope coefficient is equal to the correlation coefficient times the ratio of standard deviations:

$$
\beta_1 
= \frac{Cov(X,Y)}{Var(X)} 
= \frac{Cov(X,Y)}{(SD(X))^2} 
= \frac{Cov(X,Y)}{SD(X)}\cdot \frac{1}{SD(X)} 
= \frac{Cov(X,Y)}{SD(X)\cdot SD(Y)}\cdot \frac{SD(Y)}{SD(X)}
= Corr(X,Y) \cdot \frac{SD(Y)}{SD(X)}
$$

The solution to the minimisation problem satisfies two additional properties:

 - The mean of prediction errors is zero: 
$$
\mathbb{E}[\varepsilon] = 0
$$
 - The regressor and the error term are uncorrelated:
$$
Cov(X, \varepsilon) = 0
$$



**Applying Ordinary Least Squares to our example**

In our example, the population linear approximation can be written as:

$$
log\_hpay \ =\ \beta_0 + \beta_1 \cdot ASVAB\_pct + \varepsilon
$$

The sample OLS procedure is the sample analogue of the population least squares problem.

When implementing the procedure, we work with **residuals** instead of an error term for the difference between actual outcome and linear prediction.

The candidate residuals as a function of candidate linear prediction parameters $(b_0, b_1)$ are:

$$
u_{i}(b_0, b_1) = log\_hpay_{i} - (b_0 + b_1 \cdot ASVAB\_pct_{i}) \ \ \ (\text{for each } i \in Sample)
$$

In particular, the OLS minimisation corresponds to **minimising the mean of squared residuals**:

$$
\begin{array}{crcl}
&
(\widehat{\beta}_0, \widehat{\beta}_1) 
&=& \arg \underset{(b_0, b_1)}{\min} \  \displaystyle\frac{1}{N} \sum_{i \in Sample} \left(u_{i}(b_0, b_1)\right)^2 \\ \\
\Longleftrightarrow &
(\widehat{\beta}_0, \widehat{\beta}_1) 
&=& \arg \underset{(b_0, b_1)}{\min} \  \displaystyle\frac{1}{N} \sum_{i \in Sample} \left(log\_hpay_{i} - (b_0 + b_1 \cdot ASVAB\_pct_{i})\right)^2 
\end{array}
$$

The resulting parameter estimates are:

$$
\begin{array}{rcll}
\widehat{\beta}_0 &=& \displaystyle \frac{1}{N} \sum_{i \in Sample} log\_hpay_{i} - \widehat{\beta}_1 \cdot \frac{1}{N} \sum_{i \in \text{Sample}} ASVAB\_pct_{i} & \ \ (\text{intercept}) \\ \\
\widehat{\beta}_1 &=& \displaystyle \frac{Cov_{Sample}(ASVAB\_pct,log\_hpay)}{Var_{Sample}(ASVAB\_pct)}  &  \ \ (\text{slope coefficient})
\end{array}
$$

The estimated equation can be written as:

$$
\underbrace{log\_hpay_{i}}_{\mathclap{\substack{\text{actual outcome}}}} \ =\  \widehat{\beta}_0 + \widehat{\beta}_1 \cdot \underbrace{ASVAB\_pct_{i}}_{\mathclap{\substack{\text{regressor}}}} + \underbrace{\widehat{u}_{i}}_{\mathclap{\substack{\text{residual}}}}
$$

And the predicted value of log hourly pay for individual $i$ corresponds to:

$$
\underbrace{\widehat{log\_hpay}_{i}}_{\mathclap{\substack{\text{predicted outcome}}}} 
\ =\ \underbrace{log\_hpay_{i}}_{\mathclap{\substack{\text{actual outcome}}}} - \underbrace{\widehat{u}_{i}}_{\mathclap{\substack{\text{residual}}}} 
\ =\ \widehat{\beta}_0 + \widehat{\beta}_1 \cdot \underbrace{ASVAB\_pct_{i}}_{\mathclap{\substack{\text{regressor}}}} 

$$

The solution to the minimisation problem satisfies the sample counterparts of the two properties seen above in the case of the population OLS:

 - The mean of residuals is zero: 
$$
\displaystyle \frac{1}{N} \sum_{i \in Sample} \widehat{u}_{i} = 0
$$
 - The regressor and the residuals are uncorrelated:
$$
Cov_{Sample}(ASVAB\_pct, \widehat{u}) = 0
$$





Let us now run the OLS procedure for our example using our sample data.

We first use the `statsmodels.regression.linear_model.OLS` class directly.

To create an OLS model object, we need:

 - the array for the outcome variable $y$ (`log_hpay`)
 - the array for the regressor variable(s) $X$ (here only `ASVAB_pct`)
 - to add a constant to the regressor array $X$ in order to include an intercept in the model

We can then call the `fit()` method of the model object to estimate the regression coefficients and store the results in an object called `results`.

In [ ]:
df = merged_df.copy()

varnames = ['log_hpay', 'ASVAB_pct']

df = df[varnames].dropna().copy()
df = df.astype(float)

y = df['log_hpay']

X = df[['ASVAB_pct']]


X = sm.add_constant(X)
results = sm.OLS(y, X).fit()

# print the regression table with all model stats
# print(results.summary())

# # print all results attributes
# [attr for attr in dir(results) if not attr.startswith('_')]

An alternative approach is to specify the regression model using the formula interface `smf`. This syntax may look familiar to users of the `lm()` function in R.

In [ ]:
df = merged_df.copy()
df = df[['log_hpay', 'ASVAB_pct']].dropna().copy()

results_f = smf.ols(
    formula='log_hpay ~ ASVAB_pct',
    data=df
).fit()

# print(results_f.summary())

Let us check that the two approaches produce equivalent results:

In [ ]:
np.allclose(
    results.params,
    results_f.params
)

**Estimated regression line**

The OLS linear approximation of the conditional mean function is:

$$
\underbrace{\widehat{log\_hpay}_{i}}_{\mathclap{\substack{\text{predicted outcome}}}} \ =\ 2.94 + 0.0108 \cdot \underbrace{ASVAB\_pct_{i}}_{\mathclap{\substack{\text{regressor}}}} 
$$



Let us plot the regression line of predicted values on our scatter plot:

In [ ]:
df = merged_df.copy()

predict_df = pd.DataFrame({'ASVAB_pct': np.linspace(start=df['ASVAB_pct'].min(), stop=df['ASVAB_pct'].max(),num=100)})

slope = results_f.params['ASVAB_pct']
intercept = results_f.params['Intercept']

predict_df['log_hpay_hat'] = results_f.predict(predict_df) 
# predict_df['log_hpay_hat'] = intercept + slope * predict_df['ASVAB_pct']
fig, ax = plt.subplots()
ax.scatter(x=df['ASVAB_pct'], y=df['log_hpay'], s=2, alpha=0.5)

ax.plot(predict_df['ASVAB_pct'], predict_df['log_hpay_hat'], label=f"OLS Regression Line: log_hpay_hat = {intercept:0.2f} + {slope:0.4f} * ASVAB_pct ", color='red',linewidth=3)

ax.set_ylabel('Log Hourly Pay in 2023 (dollars, logarithmic scale)', fontsize=10)
ax.set_xlabel('ASVAB test score percentile rank in 1997')
ax.set_xlim(left=0, right=100)


ax.legend(fontsize=8)

Let us verify that the statsmodels interface gives the same values as the formulas.

In [ ]:
df = merged_df.copy()
df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').copy()
df = df.astype(float)

covariance = np.cov(m=df, rowvar=False, ddof=1)[0,1]
covariance

variance_X = df['ASVAB_pct'].var(ddof=1)
variance_X

slope_by_hand = covariance / variance_X
slope_by_hand

intercept_by_hand = df['log_hpay'].mean() - slope_by_hand * df['ASVAB_pct'].mean()
intercept_by_hand

print('Intercept "by hand":',
      intercept_by_hand,
      '\nIntercept interface:',
      results_f.params['Intercept'],
      '\n\nSlope coefficient "by hand"',
      slope_by_hand,
      '\nSlope coefficient interface:',
      results_f.params['ASVAB_pct'])


**Interpretation of parameter estimates**

Let us remind ourselves that the original objective was an approximation of the conditional mean function.

So let us plot 
 - the conditional means of log hourly pay by ASVAB test score percentile bins again 
 - together with the estimated regression line.

In [ ]:
df = merged_df.copy()
df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').copy()
df = df.astype(float)
df['ASVAB_pct_bin'] = np.floor(df['ASVAB_pct'])

plot_df = df.groupby(by='ASVAB_pct_bin').agg(mean_log_hpay = ('log_hpay', 'mean')).reset_index().copy()
plot_df = plot_df.convert_dtypes().sort_values(by='ASVAB_pct_bin')

predict_df = plot_df.rename(columns={'ASVAB_pct_bin': 'ASVAB_pct'}).copy()
predict_df['log_hpay_hat'] = results_f.predict(predict_df['ASVAB_pct'])

fig, ax = plt.subplots()


ax.scatter(x=plot_df['ASVAB_pct_bin'], y=plot_df['mean_log_hpay'], s=10)

slope = results_f.params['ASVAB_pct']
intercept = results_f.params['Intercept']
ax.plot(predict_df['ASVAB_pct'], predict_df['log_hpay_hat'], label=f"OLS Regression Line: log_hpay_hat = {intercept:0.2f} + {slope:0.4f} * ASVAB_pct ", color='red',linewidth=3)


ax.set_ylabel('Conditional mean of log hourly pay in 2023 (dollars)', fontsize=8)
ax.set_xlabel('ASVAB test score percentile rank in 1997')

ax.set_xlim(left=0, right=100)
ax.legend(fontsize=8)



**Interpretation of parameter estimates**

In our example, the estimated regression equation is:

$$
\underbrace{\widehat{log\_hpay}}_{\mathclap{\substack{\text{predicted outcome}}}}
=
2.94
+
0.0108
\cdot
\underbrace{ASVAB\_pct}_{\mathclap{\substack{\text{regressor}}}}
$$

The OLS regression line is therefore a linear approximation of the conditional mean of log hourly pay given the ASVAB test score percentile rank.

**Interpretation of the intercept**

What is the interpretation of the intercept $\widehat{\beta}_0 \approx 2.94$?

The intercept is the predicted value of log hourly pay for individuals whose ASVAB test score percentile rank is zero.

In this example, this means that the model predicts a log hourly pay of approximately 2.94 for individuals at the bottom of the ASVAB percentile rank distribution.

The meaningfulness of the intercept is often limited, especially if the regressor cannot realistically take the value zero or if zero lies outside the range of observed values.

**Interpretation of the slope coefficient**

What is the interpretation of the slope coefficient $\widehat{\beta}_1 \approx 0.0108$?

The slope coefficient is the average change in the predicted value of the outcome variable associated with a one-unit increase in the regressor.

In this example, scoring one percentile point higher on the ASVAB test is associated with an increase of approximately 0.0108 in predicted log hourly pay.

Since the outcome variable is measured in logarithms, we can approximately interpret the coefficient as a percentage change in hourly pay (after multiplying by $100$). For small values of $v$, we can use the approximation

$$
e^v \approx 1 + v\quad \Longleftrightarrow \quad \ln(1 + v) \approx v
$$

The linear prediction is

$$
\widehat{log\_hpay}
=
2.94
+
0.0108
\cdot
ASVAB\_pct.
$$

Exponentiating both sides gives the predicted hourly pay:

$$
\widehat{hpay}
=
e^{2.94 + 0.0108 \cdot ASVAB\_pct}.
$$

Now consider increasing the ASVAB test score percentile rank by one point.

The predicted proportional change in hourly pay is

$$
\frac{\widehat{hpay}_{new}-\widehat{hpay}}
{\widehat{hpay}}
=
\frac{
e^{2.94+0.0108(ASVAB\_pct+1)}
-
e^{2.94+0.0108ASVAB\_pct}
}
{e^{2.94+0.0108ASVAB\_pct}}.
$$

Using the properties of exponentials ($e^{a+b}=e^ae^b$), this simplifies to

$$
\frac{\widehat{hpay}_{new}-\widehat{hpay}}
{\widehat{hpay}}
=
e^{0.0108}-1.
$$

Using the approximation $e^v \approx 1+v$ for small values of $v$ gives

$$
e^{0.0108}-1
\approx
0.0108
=
1.08\%.
$$

Therefore, according to the estimated regression model, scoring one percentile point higher on the ASVAB test is associated with approximately **1.08% higher predicted hourly pay**, on average.

The exact proportional change is

$$
e^{0.0108}-1
\approx
1.09\%.
$$

The approximate and exact interpretations are almost identical because the estimated coefficient is small.

In [ ]:
# slope = results_f.params['ASVAB_pct']
# print(np.exp(slope) - 1, slope)

The OLS residuals are by construction uncorrelated with the regressors used in the model:

$$
Cov_{Sample}(ASVAB\_pct, \widehat{u}) = 0
$$

This is a mechanical property of the OLS estimation procedure.

This does not imply however that the residuals are mean independent from the regressors.

Let us see these properties both visually and numerically.

Let us start with a scatter plot of residuals against regressor values.

In [ ]:
df = merged_df.copy()

df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').sort_values(by='ASVAB_pct').copy()
df['log_hpay_hat'] = results_f.predict(df)
df['residuals'] = df['log_hpay'] - df['log_hpay_hat']
fig, ax = plt.subplots()
ax.scatter(x=df['ASVAB_pct'], y=df['residuals'], s=6, alpha=0.5)
ax.set_ylabel('Residuals = Actual log hourly pay $-$ Predicted log hourly pay', fontsize=8)
ax.set_xlabel('ASVAB test score percentile rank in 1997')


Let us now look plot the conditional mean residual per ASVAB test score percentile rank (regressor) bin.

In [ ]:
df = merged_df.copy()

df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').sort_values(by='ASVAB_pct').copy()
df['log_hpay_hat'] = results_f.predict(df)
df['residual'] = df['log_hpay'] - df['log_hpay_hat']


df['ASVAB_pct_bin'] = np.floor(df['ASVAB_pct'])
df_plot = df.groupby('ASVAB_pct_bin').agg(mean_residual = ('residual', 'mean')).reset_index().sort_values('ASVAB_pct_bin').copy()

fig, ax = plt.subplots()
ax.scatter(x=df_plot['ASVAB_pct_bin'], y=df_plot['mean_residual'], s=6, alpha=0.5)
ax.set_ylabel('Conditional mean residual')
ax.set_xlabel('ASVAB test score percentile rank in 1997')

The residuals are clearly not mean independent of the regressor.

However, the best linear approximation of the conditional mean of the residuals given the regressor is simply a horizontal line at zero. They are uncorrelated. This can be verified numerically:

In [ ]:
df = merged_df.copy()

df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').sort_values(by='ASVAB_pct').copy()
df['log_hpay_hat'] = results_f.predict(df)
df['residual'] = df['log_hpay'] - df['log_hpay_hat']

covariance = df['ASVAB_pct'].cov(df['residual'])
np.isclose(covariance, 0)
# df = df[['residual', 'ASVAB_pct']].dropna(axis=0, how='any').copy()
# df = df.astype(float)
# np.isclose(
#     np.cov(m=df, rowvar=False)[0,1],
#     0
# )


**Goodness of fit**

A common measure of model fit is the $R^2$ statistic.

It measures the share of the variance of the outcome variable that is explained by the model.

To understand this statistic, notice that the observed outcome can be decomposed as

$$
Y = \widehat{Y} + \widehat{u}
$$

where $\widehat{Y}$ denotes the predicted values and $\widehat{u}$ the residuals.

For our example, this decomposition corresponds to:

$$
log\_hpay_{i} 
\ =\ \widehat{log\_hpay}_{i} + \widehat{u}_{i}
$$

Moreover, because the predicted values are linear combinations of the regressors and the residuals are uncorrelated with the regressors, the fitted values and residuals are also uncorrelated:

$$
Cov_{Sample}(\widehat{Y},\widehat{u}) = 0.
$$

In our example, this becomes:
$$
Cov_{Sample}(\widehat{log\_hpay}_{i}, \widehat{u}_{i}) = 0
$$

Let us verify this numerically.

In [ ]:
df = merged_df.copy()

df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').sort_values(by='ASVAB_pct').copy()
df['log_hpay_hat'] = results_f.predict(df)
df['residual'] = df['log_hpay'] - df['log_hpay_hat']
covariance = df['log_hpay_hat'].cov(df['residual'])
np.isclose(covariance, 0)


Because the predicted values of the outcome variable and the residuals are uncorrelated, the variance of the outcome can be decomposed as follows:

$$
Var_{Sample}(Y) = Var_{Sample}(\widehat{Y} + \widehat{u}) = Var_{Sample}(\widehat{Y}) + Var_{Sample}(\widehat{u})
$$

The $R^2$ statistic corresponds to the ratio of the variance of the predicted values of the outcome variable to the variance of the outcome variable:

$$
R^2 = \frac{Var_{Sample}(\widehat{Y})}{Var_{Sample}(Y)} 
$$

In the case of the simple linear regression model, it can be shown that the $R^2$ statistic is equal to the square of the correlation coefficient between the outcome variable and the regressor:

$$
R^2 = (Corr_{Sample}(X, Y))^2
$$

In our example, the equivalence corresponds to:

$$
R^2 = \frac{Var_{Sample}(\widehat{log\_hpay})}{Var_{Sample}(log\_hpay)}  = (Corr_{Sample}(ASVAB\_pct, log\_hpay))^2
$$

Let us verify this equivalence numerically.

In [ ]:
df = merged_df.copy()

df = df[['log_hpay', 'ASVAB_pct']].dropna(axis=0, how='any').sort_values(by='ASVAB_pct').copy()
df['log_hpay_hat'] = results_f.predict(df)
R2_variance_decomposition = df['log_hpay_hat'].var() / df['log_hpay'].var()
R2_correlation_squared = (df['ASVAB_pct'].corr(df['log_hpay'])) ** 2
print('R2 variance decomposition:', R2_variance_decomposition, 
      '\nR2 correlation squared:', R2_correlation_squared)


In our sample, the simple linear regression model using the ASVAB test score percentile rank as the only regressor explains about 13.9% of the variation in log hourly pay.

Note that in the case of simple linear regression with an intercept, the slope coefficient estimate and the $R^2$ statistic are both directly related to the sample correlation coefficient.

The $R^2$ statistic is simply the squared sample correlation:

$$
R^2 = \left(Corr_{Sample}(X,Y)\right)^2
$$

The slope coefficient estimate can also be expressed in terms of the sample correlation coefficient:

$$
\widehat{\beta}_1
=
Corr_{Sample}(X,Y)
\cdot
\frac{SD_{Sample}(Y)}{SD_{Sample}(X)}
$$

